# Notebook 08 — Spatial ABM: Movement and Chemotaxis

**Module:** 15 — Simulation and Agent-Based Modeling  
**Tier:** 2 — Working competence  
**Notebook:** 8 of 12  
**Time estimate:** 75 minutes

> Biological agents move. Neutrophils chase bacteria through chemical gradients.
> Slime mold cells aggregate by following cAMP waves. Neural growth cones navigate
> by sensing axon guidance cues. This notebook models individual movement rules
> and shows how simple movement strategies produce emergent spatial structure.

---
## Step 1 — Motivation

The spatial distribution of organisms is not random — it is structured by movement
and interaction. A neutrophil's ability to find a bacterium is the difference
between clearing an infection and sepsis. *Dictyostelium discoideum* aggregates
into a fruiting body when starving — an emergent collective structure from simple
cAMP-following individual movement rules. Modelling this at the individual level
reveals how the collective arises.

---
## Step 2 — Intuition

**Random walk:** the null model of movement. At each step, move in a random
direction. Mean squared displacement grows linearly: $\langle r^2 \rangle = 2dDt$
where d=spatial dimension, D=diffusion coefficient. Not efficient for finding targets.

**Biased random walk (chemotaxis):** move preferentially in the direction of
increasing chemical gradient (chemoattractant) or decreasing (chemorepellent).
Can be modelled as run-and-tumble (bacteria: long runs, brief random tumbles
suppressed in favourable gradient direction).

**Lévy flight:** move with a step length drawn from a heavy-tailed distribution
P(l) ~ l^{-μ}, 1 < μ ≤ 3. Produces occasional very long steps between local
searches. Theoretically optimal for finding sparse randomly-distributed targets
(Lévy foraging hypothesis — controversial but influential).

**Collective chemotaxis (Dictyostelium):** individual cells secrete cAMP,
follow cAMP gradients, and degrade it with phosphodiesterase. The coupling
between secretion, diffusion, and following produces a propagating spiral wave
that herds cells toward aggregation centres.

---
## Step 3 — Biological Background

**Bacterial chemotaxis (E. coli run-and-tumble):**
E. coli alternates between runs (flagella rotate CCW, move forward) and tumbles
(flagella rotate CW, random new direction). The flagellar motor responds to the
rate of change of chemoattractant concentration over time — not the absolute
level. If concentration is increasing, suppress tumbles (run longer). If
decreasing, tumble more. This implements gradient ascent without explicit
spatial sensing — comparing past and present to infer gradient direction.

**Neutrophil chemotaxis (eukaryotic):** neutrophils use PIP2/PIP3 polarity — the
front accumulates PIP3 (activates Rac, actin protrusion) and the back accumulates
PIP2 (activates ROCK, myosin retraction). The resulting force imbalance propels
the cell toward the chemoattractant (fMLP, C5a, IL-8).

**Dictyostelium aggregation:**
1. Starvation triggers cAMP secretion in leader cells
2. Neighbouring cells detect cAMP, amplify it, relay the wave
3. Cells move toward increasing cAMP (runs toward wave crest)
4. cAMP is degraded by extracellular phosphodiesterase (creates a refractory gradient)
5. Result: centripetal waves → aggregation streams → fruiting body

---
## Step 4 — Mathematical Explanation

**Diffusion (random walk in 2D):**
$$\langle r^2(t) \rangle = 4Dt, \quad D = \frac{\ell^2}{4\tau}$$
where $\ell$ = step length, $\tau$ = step duration.

**Chemotaxis gradient sensing model:**
$$\theta_{t+1} = \theta_t + \Delta\theta, \quad \Delta\theta \sim \mathcal{N}(\theta^*, \sigma^2)$$
where $\theta^* = \text{arg max}_{\theta} \nabla C \cdot \hat{e}(\theta)$ is the
direction of steepest ascent, and $\sigma$ controls the noise level.
For a linear gradient $C(x,y) = C_0 + gx$, $\theta^* = 0$ (move in +x).

**Lévy flight step length distribution:**
$$P(l) \propto l^{-\mu}, \quad 1 < \mu \leq 3$$
Implemented via inverse CDF: $l = l_{\min} \cdot u^{-1/(\mu-1)}$ where $u \sim U(0,1)$.
MSD grows superdiffusively: $\langle r^2 \rangle \sim t^{2/\gamma}$ with $\gamma = \mu - 1$
for 1 < μ < 3; ballistic ($\sim t^2$) for μ ≤ 2.

**Chemotaxis + diffusion (Keller-Segel PDE):**
$$\frac{\partial \rho}{\partial t} = D_\rho \nabla^2 \rho - \chi \nabla \cdot (\rho \nabla C)$$
$$\frac{\partial C}{\partial t} = D_C \nabla^2 C + \alpha \rho - \beta C$$
where $\chi$ = chemotactic sensitivity, $\alpha$ = secretion rate, $\beta$ = degradation rate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d

rng = np.random.default_rng(42)

DOMAIN = 100.0  # simulation box size

# ---- 1. Pure random walk ----
def random_walk_2d(n_agents, n_steps, step_size, rng):
    """2D random walk with periodic boundary."""
    pos = rng.uniform(0, DOMAIN, (n_agents, 2))
    positions = [pos.copy()]
    for _ in range(n_steps):
        angles = rng.uniform(0, 2*np.pi, n_agents)
        pos[:, 0] = (pos[:, 0] + step_size * np.cos(angles)) % DOMAIN
        pos[:, 1] = (pos[:, 1] + step_size * np.sin(angles)) % DOMAIN
        positions.append(pos.copy())
    return np.array(positions)

# ---- 2. Biased random walk (chemotaxis) ----
def linear_gradient(x, y, g=0.05, C0=0.0):
    """Linear chemoattractant concentration gradient along x."""
    return C0 + g * x

def chemotaxis_walk(n_agents, n_steps, step_size, grad_strength, noise_sigma, rng):
    """
    Biased random walk: move toward gradient with noise.
    grad_strength: weight of gradient direction vs. noise
    """
    pos = rng.uniform(0, DOMAIN/2, (n_agents, 2))  # start left half
    positions = [pos.copy()]
    for _ in range(n_steps):
        # Gradient direction = +x (steepest ascent of linear C)
        grad_dir = np.zeros((n_agents, 2))
        grad_dir[:, 0] = 1.0  # gradient points in +x
        # Noisy movement: bias toward gradient
        angles = rng.uniform(0, 2*np.pi, n_agents)
        noise_dir = np.stack([np.cos(angles), np.sin(angles)], axis=1)
        move = grad_strength * grad_dir + (1 - grad_strength) * noise_dir
        norms = np.linalg.norm(move, axis=1, keepdims=True)
        move = move / norms * step_size
        pos = (pos + move) % DOMAIN
        positions.append(pos.copy())
    return np.array(positions)

# ---- 3. Lévy flight ----
def levy_walk(n_agents, n_steps, mu=1.5, l_min=0.5, rng=rng):
    """Lévy walk with power-law step length distribution."""
    pos = rng.uniform(0, DOMAIN, (n_agents, 2))
    positions = [pos.copy()]
    for _ in range(n_steps):
        angles = rng.uniform(0, 2*np.pi, n_agents)
        u = rng.uniform(0, 1, n_agents)
        step_lengths = l_min * u**(-1/(mu-1))  # inverse CDF
        step_lengths = np.minimum(step_lengths, DOMAIN/2)  # cap at domain/2
        pos[:, 0] = (pos[:, 0] + step_lengths * np.cos(angles)) % DOMAIN
        pos[:, 1] = (pos[:, 1] + step_lengths * np.sin(angles)) % DOMAIN
        positions.append(pos.copy())
    return np.array(positions)

# Run simulations
N_AGENTS = 100; N_STEPS = 200; STEP = 1.0
print('Running random walk...')
pos_rw = random_walk_2d(N_AGENTS, N_STEPS, STEP, rng)
print('Running chemotaxis walk...')
pos_ct = chemotaxis_walk(N_AGENTS, N_STEPS, STEP, grad_strength=0.7, noise_sigma=0.5, rng=rng)
print('Running Lévy walk...')
pos_lv = levy_walk(N_AGENTS, N_STEPS, mu=1.5, rng=rng)

# Mean squared displacement for each
def msd(positions, max_lag=100):
    """Compute MSD as function of lag time."""
    msds = []
    for lag in range(1, max_lag+1):
        disp = positions[lag:] - positions[:len(positions)-lag]
        # Handle periodic boundary (unwrapping)
        disp[disp > DOMAIN/2]  -= DOMAIN
        disp[disp < -DOMAIN/2] += DOMAIN
        msds.append(np.mean(disp**2))
    return np.array(msds)

msd_rw = msd(pos_rw)
msd_ct = msd(pos_ct)
msd_lv = msd(pos_lv)
print('MSD computed.')

In [ ]:
# ---- Slime mold aggregation (simplified Keller-Segel) ----
def run_keller_segel(N=60, n_steps=300, n_cells=200,
                      D_C=0.5, alpha=0.5, beta_deg=0.1,
                      chi=1.5, D_rho=0.05, seed=42):
    """
    Simplified Keller-Segel on discrete grid.
    Cells are point particles; concentration C on a grid.
    """
    rng_ks = np.random.default_rng(seed)
    # Initial conditions: cells uniformly distributed
    cell_pos = rng_ks.integers(0, N, (n_cells, 2))
    C = np.zeros((N, N), dtype=float)  # cAMP concentration
    # Diffusion kernel (finite differences)
    lap_kernel = np.array([[0, 1, 0],[1,-4,1],[0,1,0]])

    snapshots = []
    for t in range(n_steps):
        # Cells secrete cAMP at their position
        for r, c in cell_pos:
            C[r, c] += alpha
        # cAMP diffusion + degradation
        lap_C = convolve2d(C, lap_kernel, mode='same', boundary='wrap')
        C = np.clip(C + D_C * lap_C - beta_deg * C, 0, None)
        # Cells move: sense gradient of C, move up-gradient with noise
        for i, (r, c) in enumerate(cell_pos):
            # Local gradient (finite differences)
            dCdx = (C[r, (c+1)%N] - C[r, (c-1)%N]) / 2
            dCdy = (C[(r+1)%N, c] - C[(r-1)%N, c]) / 2
            grad = np.array([dCdy, dCdx])
            grad_norm = np.linalg.norm(grad)
            if grad_norm > 0:
                grad = grad / grad_norm
            noise = rng_ks.standard_normal(2) * 0.3
            move = chi * grad + noise
            # Clamp to integer step
            dr = int(np.clip(np.round(move[0]), -1, 1))
            dc = int(np.clip(np.round(move[1]), -1, 1))
            cell_pos[i] = [(r + dr) % N, (c + dc) % N]
        if t % 50 == 0:
            snap_density = np.zeros((N, N), dtype=float)
            for r, c in cell_pos:
                snap_density[r, c] += 1
            snapshots.append((t, snap_density.copy(), C.copy()))
    return snapshots, cell_pos

print('Running Keller-Segel slime mold aggregation...')
ks_snapshots, ks_final = run_keller_segel(N=60, n_steps=300, n_cells=150)
print('Done.')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Panel A: Trajectories of 5 agents for each walk type
ax = axes[0, 0]
for i in range(5):
    traj = pos_rw[:, i, :]
    ax.plot(traj[:, 0], traj[:, 1], alpha=0.6, lw=0.8, color='steelblue')
    ax.plot(traj[0, 0], traj[0, 1], 'o', color='steelblue', ms=4)
ax.set_xlim(0, DOMAIN); ax.set_ylim(0, DOMAIN)
ax.set_title('A. Random walk\n(5 agent trajectories, 200 steps)')
ax.set_aspect('equal')

# Panel B: Chemotaxis trajectories
ax = axes[0, 1]
for i in range(5):
    traj = pos_ct[:, i, :]
    ax.plot(traj[:, 0], traj[:, 1], alpha=0.6, lw=0.8, color='tomato')
    ax.plot(traj[0, 0], traj[0, 1], 'o', color='tomato', ms=4)
# Show gradient direction
for gx in range(10, 100, 20):
    ax.annotate('', xy=(gx+5, 50), xytext=(gx, 50),
                   arrowprops=dict(arrowstyle='->', color='grey', lw=1))
ax.set_xlim(0, DOMAIN); ax.set_ylim(0, DOMAIN)
ax.set_title('B. Chemotaxis walk (gradient → +x)\n(agents drift toward high concentration)')
ax.set_aspect('equal')

# Panel C: MSD comparison
ax = axes[0, 2]
lag_vals = np.arange(1, 101)
ax.loglog(lag_vals, msd_rw, 'steelblue', lw=2, label='Random walk')
ax.loglog(lag_vals, msd_ct, 'tomato', lw=2, label='Chemotaxis')
ax.loglog(lag_vals, msd_lv, 'green', lw=2, label='Lévy flight (μ=1.5)')
# Reference lines
ax.loglog(lag_vals, lag_vals * msd_rw[0], 'k:', lw=1, label='~ t (diffusive)')
ax.loglog(lag_vals, lag_vals**1.5 * 0.5, 'k--', lw=1, label='~ t^1.5 (superdiffusive)')
ax.set_xlabel('Lag time'); ax.set_ylabel('MSD')
ax.set_title('C. Mean squared displacement\n(log-log: slope = diffusion exponent)')
ax.legend(fontsize=7)

# Panel D: Final agent positions for each walk type
ax = axes[1, 0]
ax.scatter(pos_rw[-1, :, 0], pos_rw[-1, :, 1], s=10, alpha=0.6,
             color='steelblue', label='Random walk')
ax.scatter(pos_ct[-1, :, 0], pos_ct[-1, :, 1], s=10, alpha=0.6,
             color='tomato', label='Chemotaxis')
ax.scatter(pos_lv[-1, :, 0], pos_lv[-1, :, 1], s=10, alpha=0.6,
             color='green', label='Lévy')
ax.set_xlim(0, DOMAIN); ax.set_ylim(0, DOMAIN)
ax.set_title('D. Final agent positions (t=200)\n(chemotaxis agents clustered right)')
ax.legend(fontsize=8)
ax.set_aspect('equal')

# Panel E: Keller-Segel cell density snapshots
ax = axes[1, 1]
if len(ks_snapshots) >= 3:
    t0, d0, c0 = ks_snapshots[0]
    t1, d1, c1 = ks_snapshots[len(ks_snapshots)//2]
    t2, d2, c2 = ks_snapshots[-1]
    combined = np.concatenate([d0, d1, d2], axis=1)
    ax.imshow(combined, cmap='hot', aspect='auto')
    ax.axvline(60, color='white', lw=1)
    ax.axvline(120, color='white', lw=1)
    ax.set_xticks([30, 90, 150])
    ax.set_xticklabels([f't={t0}', f't={t1}', f't={t2}'])
    ax.set_title('E. Keller-Segel aggregation\n(cell density at 3 time points)')
    ax.axis('on')
else:
    axes[1, 1].text(0.5, 0.5, 'Run KS simulation', ha='center', va='center',
                       transform=axes[1,1].transAxes)

# Panel F: cAMP concentration field (final)
ax = axes[1, 2]
if len(ks_snapshots) >= 1:
    _, _, c_final = ks_snapshots[-1]
    im = ax.imshow(c_final, cmap='YlOrRd', origin='lower', interpolation='bilinear')
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title('F. cAMP concentration field (final)\n(high near cell aggregates)')
    ax.axis('off')

plt.suptitle('Module 15 NB08: Spatial ABM — Movement and Chemotaxis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('spatial_abm_movement.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement the run-and-tumble model for bacterial chemotaxis: during a run,
   continue in the current direction for a run time drawn from Exp(λ). Tumble
   with rate λ that decreases when the concentration gradient is positive
   (fewer tumbles in favourable gradient). Compare to the biased random walk.
2. Show that the MSD of a random walk scales linearly with time (t¹) and that
   the MSD of a Lévy flight with μ=1.5 scales faster. Extract the scaling
   exponent by fitting a line to the log-log MSD plot.
3. Add cell-cell repulsion to the Keller-Segel model: if a cell would move to
   a grid point that already has >5 cells, reduce the probability of that move.
   How does it affect the aggregation pattern?

---
## Step 10 — Quiz

1. What is the Keller-Segel model? Name its four terms and the biological
   process each represents.
2. How does E. coli implement gradient sensing without directly measuring the
   spatial gradient? What does it measure instead?
3. What is the mean squared displacement (MSD)? How can you distinguish diffusive,
   subdiffusive, and superdiffusive motion from the MSD slope on a log-log plot?
4. What is the Lévy foraging hypothesis? Describe one piece of empirical evidence
   for and one piece of evidence against it.

---
## Step 12 — Reflection

> *[In one paragraph: explain how the Keller-Segel model produces aggregation
> from individual chemotaxis. Walk through: (1) cells secrete cAMP, (2) cAMP
> diffuses and degrades, (3) cells move up the cAMP gradient. Why does a positive
> feedback loop form? What prevents the cells from collapsing into a single
> point (mathematical blow-up)?]*

---
*Next: `09_parameter_estimation.ipynb`*